In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 43. Week 29 — Conjugacy, prior predictive checks, and Bayesian regression

## 学習目標

- normal-normal conjugacyをprecisionで導出できる
- prior predictiveでscale mismatchを発見できる
- Normal–Inverse-Gamma regressionのposterior predictiveを作れる
- validation coverageとwidthをpoint errorから分けられる

## 前提知識

- Gaussian likelihood、linear regression
- B7の5公表日先target

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 43


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Conjugate normal mean

既知観測分散 (sigma^2)、prior (mu\sim N(m_0,s_0^2)) なら

$$
s_n^{-2}=s_0^{-2}+n\sigma^{-2},\qquad
m_n=s_n^2\left(s_0^{-2}m_0+\sigma^{-2}\sum_i y_i\right).
$$

In [4]:
horizon = 5
origins = np.arange(curve_yields.shape[0] - horizon)
targets = (curve_yields[origins + horizon] - curve_yields[origins]) * 100.0
target_dates = curve_dates[origins + horizon]
training_rows = target_dates <= train_end_date
validation_rows = (target_dates > train_end_date) & (target_dates <= validation_end_date)

ten_year_training = targets[training_rows, 3]
observation_variance = float(np.var(ten_year_training, ddof=1))
posterior_rows = []
for prior_sd in [1.0, 5.0, 20.0]:
    posterior = qt.normal_mean_posterior(
        ten_year_training,
        observation_variance=observation_variance,
        prior_mean=0.0,
        prior_variance=prior_sd**2,
    )
    lower, upper = posterior.predictive_interval(0.9)
    posterior_rows.append(
        {"prior_sd_bp": prior_sd, "posterior_mean_bp": posterior.mean, "posterior_sd_bp": np.sqrt(posterior.variance), "predictive_lower_bp": lower, "predictive_upper_bp": upper}
    )
display(pd.DataFrame(posterior_rows))

,prior_sd_bp,posterior_mean_bp,posterior_sd_bp,predictive_lower_bp,predictive_upper_bp
0,1.0,-0.203123,0.230741,-16.052005,15.645759
1,5.0,-0.214064,0.236874,-16.063191,15.635063
2,20.0,-0.214515,0.237123,-16.063652,15.634622


## 2. Bayesian linear regression

$$
y\mid\beta,\sigma^2\sim N(X\beta,\sigma^2I),\quad
\beta\mid\sigma^2\sim N(0,\sigma^2\Lambda_0^{-1}),\quad
\sigma^2\sim \operatorname{InvGamma}(a_0,b_0).
$$

featuresはforecast originのcurve levelと直近1公表日のcurve change。標準化parameterはtrainingだけで固定する。

In [5]:
raw_features = np.column_stack(
    [curve_yields[origins], np.vstack([np.zeros(5), np.diff(curve_yields, axis=0)])[origins] * 100.0]
)
feature_mean = raw_features[training_rows].mean(axis=0)
feature_scale = raw_features[training_rows].std(axis=0, ddof=1)
standardized = (raw_features - feature_mean) / feature_scale
design = np.column_stack([np.ones(standardized.shape[0]), standardized])

regression = qt.fit_bayesian_linear_regression(
    design[training_rows], targets[training_rows, 3], prior_precision=1.0, prior_shape=2.0, prior_scale=25.0
)
validation_predictive = qt.bayesian_linear_predictive(regression, design[validation_rows])
lower, upper = validation_predictive.interval(0.9)
actual = targets[validation_rows, 3]
coverage = np.mean((actual >= lower) & (actual <= upper))
display(
    pd.DataFrame(
        [
            {
                "validation_rmse_bp": np.sqrt(np.mean((actual - validation_predictive.mean) ** 2)),
                "validation_coverage_90": coverage,
                "mean_interval_width_bp": np.mean(upper - lower),
                "random_walk_rmse_bp": np.sqrt(np.mean(actual**2)),
            }
        ]
    )
)

fig = go.Figure()
validation_dates = target_dates[validation_rows]
fig.add_scatter(x=validation_dates, y=actual, name="actual", mode="lines")
fig.add_scatter(x=validation_dates, y=validation_predictive.mean, name="posterior predictive mean", mode="lines")
fig.add_scatter(x=validation_dates, y=upper, name="90% upper", mode="lines", line={"width": 0})
fig.add_scatter(x=validation_dates, y=lower, name="90% interval", mode="lines", fill="tonexty", line={"width": 0})
fig.update_layout(title="Validation posterior predictive: five-publication 10y change", yaxis_title="Change (bp)", template="plotly_white")
fig.show()

,validation_rmse_bp,validation_coverage_90,mean_interval_width_bp,random_walk_rmse_bp
0,16.630609,0.650823,31.574337,15.322493


## 3. 失敗モード

- prior predictiveを観測dataでfitしてからpriorと呼ぶ
- training外でfeature standardizationをfitする
- posterior meanのRMSEだけでBayesian modelを評価する
- nominal 90%だけを見て幅を隠す
- Gaussian predictive tailを保証されたtail riskと呼ぶ

## 4. 段階別演習

### 基礎

1. precision-weighted posterior meanを再計算せよ。
2. prior scaleごとのposterior sensitivityを説明せよ。

### 標準

3. tenorごとにBayesian regressionをfitしcoverageを比較せよ。
4. validationをmethodology break前後に分けよ。

### 研究

5. Student-t likelihoodへ拡張した場合のrobustness/estimand contractを書け。

## 5. Exit Criteria

- [ ] prior、likelihood、posterior、predictiveを分けた
- [ ] prior predictiveのscaleを実データ単位で監査した
- [ ] standardizationをtrainingに限定した
- [ ] validation RMSE、coverage、widthを報告した
- [ ] Gaussian tail assumptionを明記した

## 6. 出典


- [Gelman et al., Bayesian Data Analysis, 3rd ed.](https://sites.stat.columbia.edu/gelman/book/)
- [Gelman et al., Bayesian Workflow](https://arxiv.org/abs/2011.01808)
- [Vehtari, Gelman, and Gabry, Practical Bayesian model evaluation](https://doi.org/10.1007/s11222-016-9696-4)